## Carga y preparación

In [12]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds, differential_evolution
from sklearn.preprocessing import MinMaxScaler
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

RUTA_DATOS = "datos_tratamientos_detalle_con_rating_capacidad.xlsx"
RUTA_FORECAST = "forecast_por_serie_H1_H4.xlsx"

UMBRALES_VALORACION = [3.5, 3.7, 3.8, 4.0]
UMBRAL_INICIAL = 3.7

df = pd.read_excel(RUTA_DATOS)
forecast = pd.read_excel(RUTA_FORECAST)



In [13]:
n_rating_missing = df["rating_global"].isna().sum()
df["Valoracion"] = df["rating_global"].fillna(3.0)

cols_forecast = [
    "PREDICCION_H1", "PREDICCION_H2",
    "PREDICCION_H3", "PREDICCION_H4"
]
forecast[cols_forecast] = forecast[cols_forecast].apply(pd.to_numeric, errors="coerce").fillna(0)

forecast["DemandaForecast"] = forecast[cols_forecast].sum(axis=1)
forecast_completo = forecast.copy()
demanda_total_forecast = forecast_completo["DemandaForecast"].sum()
# Para la optimización, el volumen esperado se transforma en servicios
# operativos enteros mediante redondeo al servicio entero más cercano. Conservamos el forecast decimal
# original para su interpretación.
forecast["DemandaOperativa"] = np.rint(forecast["DemandaForecast"]).astype(int)
forecast = forecast[forecast["DemandaOperativa"] > 0].copy()



In [14]:
# Una serie objetivo queda definida por tratamiento + municipio.
series_objetivo = forecast[
    ["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO",
     "DESCPROCED", "DESCMUNICIPIO",
     "PREDICCION_H1", "PREDICCION_H2",
     "PREDICCION_H3", "PREDICCION_H4",
     "DemandaForecast", "DemandaOperativa"]
].drop_duplicates()

# El histórico se restringe exclusivamente a esas series.
df_objetivo = df.merge(
    series_objetivo[["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO"]],
    left_on=["CODIGO", "CODIGO_MUNICIPIO"],
    right_on=["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO"],
    how="inner"
)



In [15]:
modelo = (
    df_objetivo.groupby(
        [
            "CODIGO_MUNICIPIO", "DESCMUNICIPIO",
            "CODPROVEEDOR", "NOMBREPROVEEDOR",
            "CODIGO", "DESCPROCED"
        ],
        as_index=False
    )
    .agg(
        CosteMedio=("SUMA", "mean"),
        NumCasos=("SUMA", "count"),
        Valoracion=("Valoracion", "mean"),
        CapacidadAnual=("CAPACIDAD_ANUAL", "first")
    )
)

# La capacidad anual es la capacidad real declarada para cada combinación
# proveedor + municipio + tratamiento.
modelo["Capacidad4Semanas"] = (
    modelo["CapacidadAnual"] * 4 / 52
)




In [20]:
ciudades = (
    series_objetivo[["CODIGO_MUNICIPIO", "DESCMUNICIPIO"]]
    .drop_duplicates()
    .sort_values(["DESCMUNICIPIO", "CODIGO_MUNICIPIO"])
)

def tratamientos_de_ciudad(codigo_municipio):
    return (
        series_objetivo[series_objetivo["CODIGO_MUNICIPIO"] == codigo_municipio]
        [["CODIGO_TRATAMIENTO", "DESCPROCED"]]
        .drop_duplicates()
        .sort_values(["DESCPROCED", "CODIGO_TRATAMIENTO"])
    )

ciudad_inicial = ciudades.iloc[0]["CODIGO_MUNICIPIO"]
tratamientos_iniciales = tratamientos_de_ciudad(ciudad_inicial)

selector_ciudad = widgets.Dropdown(
    options=[(f"{n} - {c}", c) for c, n in ciudades.itertuples(index=False, name=None)],
    value=ciudad_inicial,
    description="Ciudad:",
    layout=widgets.Layout(width="800px")
)

selector_tratamiento = widgets.Dropdown(
    options=[(f"{n} - {c}", c) for c, n in tratamientos_iniciales.itertuples(index=False, name=None)],
    value=tratamientos_iniciales.iloc[0]["CODIGO_TRATAMIENTO"],
    description="Tratamiento:",
    layout=widgets.Layout(width="800px")
)

selector_umbral = widgets.Dropdown(
    options=[(f"{u:.1f}", u) for u in UMBRALES_VALORACION],
    value=UMBRAL_INICIAL,
    description="Valoración mínima:",
    layout=widgets.Layout(width="300px")
)

def actualizar_tratamientos(change):
    tratamientos = tratamientos_de_ciudad(change["new"])
    selector_tratamiento.options = [
        (f"{n} - {c}", c)
        for c, n in tratamientos.itertuples(index=False, name=None)
    ]
    if len(tratamientos):
        selector_tratamiento.value = tratamientos.iloc[0]["CODIGO_TRATAMIENTO"]

selector_ciudad.observe(actualizar_tratamientos, names="value")

display(selector_ciudad)
display(selector_tratamiento)
display(selector_umbral)

Dropdown(description='Ciudad:', layout=Layout(width='800px'), options=(('CHACAO - 015-007-007', '015-007-007')…

Dropdown(description='Tratamiento:', layout=Layout(width='800px'), options=(('MEDICINA INTERNA CONSULTA 1A VEZ…

Dropdown(description='Valoración mínima:', index=1, layout=Layout(width='300px'), options=(('3.5', 3.5), ('3.7…

## Funciones

In [17]:
# UTILIDADES

def normalizar_minmax(grupo, invertir=False):
    """
    Normalización 0-1 dentro de cada serie ciudad + tratamiento.
    Si invertir=True, los valores menores reciben mayor puntuación.
    """
    minimo = grupo.min()
    maximo = grupo.max()

    if np.isclose(minimo, maximo):
        resultado = pd.Series(1.0, index=grupo.index)
    else:
        resultado = (grupo - minimo) / (maximo - minimo)

    if invertir:
        resultado = 1 - resultado

    return resultado


# -------------------------------------------------------------------------
# Variables de puntuación
# -------------------------------------------------------------------------
# La demanda se incorpora a la tabla de proveedores para poder medir la
# capacidad en relación con el volumen esperado, no de forma absoluta.
modelo = modelo.merge(
    series_objetivo[
        [
            "CODIGO_TRATAMIENTO",
            "CODIGO_MUNICIPIO",
            "DemandaForecast",
            "DemandaOperativa"
        ]
    ].drop_duplicates(),
    left_on=["CODIGO", "CODIGO_MUNICIPIO"],
    right_on=["CODIGO_TRATAMIENTO", "CODIGO_MUNICIPIO"],
    how="left"
)

modelo["Capacidad4Semanas"] = (
    modelo["CapacidadAnual"] * 4 / 52
)

# CosteNormalizado: 0 = más barato, 1 = más caro.
# ScoreCoste: 1 = más barato, 0 = más caro.
modelo["CosteNormalizado"] = (
    modelo.groupby(["CODIGO_MUNICIPIO", "CODIGO"])["CosteMedio"]
    .transform(normalizar_minmax)
)

modelo["ScoreCoste"] = (
    1 - modelo["CosteNormalizado"]
)

modelo["ValoracionNormalizada"] = (
    modelo.groupby(["CODIGO_MUNICIPIO", "CODIGO"])["Valoracion"]
    .transform(normalizar_minmax)
)

modelo["CapacidadRelativa"] = (
    modelo["Capacidad4Semanas"]
    / modelo["DemandaForecast"].replace(0, np.nan)
)

modelo["CapacidadRelativa"] = (
    modelo["CapacidadRelativa"].replace([np.inf, -np.inf], np.nan).fillna(0)
)

modelo["ScoreCapacidad"] = (
    modelo.groupby(["CODIGO_MUNICIPIO", "CODIGO"])["CapacidadRelativa"]
    .transform(normalizar_minmax)
)

print(f"Combinaciones ciudad-tratamiento-proveedor: {len(modelo):,}")
print(
    f"Combinaciones con capacidad anual conocida: "
    f"{modelo['CapacidadAnual'].notna().sum():,}"
)
print(
    f"Combinaciones con capacidad anual ausente: "
    f"{modelo['CapacidadAnual'].isna().sum():,}"
)

# DATOS

def obtener_datos(ciudad, tratamiento):
    datos = modelo[
        (modelo["CODIGO_MUNICIPIO"] == ciudad)
        & (modelo["CODIGO"] == tratamiento)
    ].copy()

    datos = datos.dropna(
        subset=["CapacidadAnual", "Capacidad4Semanas"]
    ).copy()

    if datos.empty:
        raise ValueError(
            "No existe una serie objetivo con capacidad conocida "
            "para la ciudad y tratamiento seleccionados."
        )

    return datos


def obtener_forecast(ciudad, tratamiento):
    f = forecast[
        (forecast["CODIGO_MUNICIPIO"] == ciudad)
        & (forecast["CODIGO_TRATAMIENTO"] == tratamiento)
    ].copy()

    if f.empty:
        raise ValueError("No existe forecast para esa serie.")

    return f.iloc[0]

def crear_capacidad_maxima(datos):
    """Adapta la capacidad anual real al horizonte de cuatro semanas."""
    datos = datos.copy()
    datos["CapacidadMaxima"] = datos["Capacidad4Semanas"]
    return datos


def demanda_forecast(ciudad, tratamiento):
    f = forecast[
        (forecast["CODIGO_MUNICIPIO"] == ciudad)
        & (forecast["CODIGO_TRATAMIENTO"] == tratamiento)
    ].copy()

    if f.empty:
        raise ValueError("No existe forecast para esa serie.")

    return int(f.iloc[0]["DemandaOperativa"])

# OPTIMIZACIÓN

def optimizar_coste(
    datos,
    demanda,
    umbral_valoracion
):
    """
    MILP de minimización de coste.

    Para cada proveedor se decide cuántos servicios recibe (x_i).
    Se minimiza el coste total respetando:
        - satisfacción exacta de la demanda operativa;
        - capacidad disponible;
        - valoración media >= umbral seleccionado.

    Este es el objetivo económico principal del MILP. La valoración
    actúa como restricción de calidad y no como parte de la función objetivo.
    """
    datos = crear_capacidad_maxima(datos).reset_index(drop=True)
    n = len(datos)
    demanda = int(demanda)

    if n == 0 or demanda <= 0:
        return {
            "Factible": False,
            "Mensaje": "Demanda no válida o no existen proveedores.",
            "CosteTotal": np.nan,
            "ValoracionMedia": np.nan,
            "SinAsignar": demanda,
            "Asignaciones": pd.DataFrame()
        }

    costes = datos["CosteMedio"].to_numpy(dtype=float)
    valoraciones = datos["Valoracion"].to_numpy(dtype=float)
    capacidades = np.floor(
        datos["CapacidadMaxima"].to_numpy(dtype=float) + 1e-9
    )

    # Función objetivo: minimizar el coste total.
    c = costes

    # Restricciones lineales:
    # 1) atender exactamente toda la demanda;
    # 2) mantener la valoración media por encima del umbral.
    A = np.vstack([
        np.ones(n),
        valoraciones
    ])

    lb = np.array([
        demanda,
        demanda * umbral_valoracion
    ], dtype=float)

    ub = np.array([
        demanda,
        np.inf
    ], dtype=float)

    resultado = milp(
        c=c,
        integrality=np.ones(n),
        bounds=Bounds(np.zeros(n), capacidades),
        constraints=LinearConstraint(A, lb, ub),
        options={"time_limit": 30}
    )

    if not resultado.success:
        return {
            "Factible": False,
            "Mensaje": resultado.message,
            "CosteTotal": np.nan,
            "ValoracionMedia": np.nan,
            "SinAsignar": demanda,
            "Asignaciones": pd.DataFrame()
        }

    x = np.rint(resultado.x).astype(int)

    coste_total = np.sum(x * costes)
    valoracion_media = np.sum(x * valoraciones) / demanda

    asignaciones = datos.loc[
        x > 0,
        [
            "CODPROVEEDOR",
            "NOMBREPROVEEDOR",
            "CosteMedio",
            "Valoracion",
            "CapacidadAnual",
            "Capacidad4Semanas"
        ]
    ].copy()

    asignaciones["Servicios"] = x[x > 0]
    asignaciones["CosteTotal"] = (
        asignaciones["Servicios"]
        * asignaciones["CosteMedio"]
    )

    asignaciones = asignaciones.sort_values(
        "Servicios",
        ascending=False
    )

    return {
        "Factible": True,
        "Mensaje": resultado.message,
        "CosteTotal": coste_total,
        "ValoracionMedia": valoracion_media,
        "SinAsignar": 0,
        "Asignaciones": asignaciones
    }

def obtener_cuotas_objetivo(
    datos,
    demanda,
    umbral,
):
    """
    Obtiene las cuotas objetivo del MILP de utilidad.
    Importante: estas cuotas NO se utilizan como filtro del ranking.
    """
    opt = optimizar_coste(
        datos,
        demanda,
        umbral,
        )

    if not opt["Factible"]:
        return pd.DataFrame()

    cuotas = opt["Asignaciones"].copy()

    cuotas["CuotaObjetivo"] = (
        cuotas["Servicios"]
        / cuotas["Servicios"].sum()
    )

    return cuotas[
        [
            "CODPROVEEDOR",
            "NOMBREPROVEEDOR",
            "Servicios",
            "CuotaObjetivo"
        ]
    ]

# RANKING

def calcular_indice(
    data,
    peso_coste=0.25,
    peso_valoracion=0.45,
    peso_capacidad=0.20,
    peso_cuota=0.10
):
    """
    Ranking multicriterio. Todos los componentes son puntuaciones
    donde un valor mayor representa una situación más favorable.
    """
    return (
        peso_coste * data["ScoreCoste"]
        + peso_valoracion * data["ValoracionNormalizada"]
        + peso_capacidad * data["ScoreCapacidad"]
        + peso_cuota * data["ScoreCuotaMILP"]
    )


asignaciones_operativas = {}


def registrar_asignacion(proveedor):
    asignaciones_operativas[proveedor] = (
        asignaciones_operativas.get(proveedor, 0) + 1
    )

def obtener_ranking_web(
    ciudad,
    tratamiento,
    umbral,
    cuotas_objetivo=None,
    peso_coste=0.25,
    peso_valoracion=0.45,
    peso_capacidad=0.20,
    peso_cuota=0.10,
    incluir_desviacion=False
):
    """
    Ranking web.

    Se mantienen TODOS los proveedores viables:
        Valoracion >= umbral
        Capacidad4Semanas > 0

    Las cuotas MILP solo aportan una puntuación; no excluyen proveedores.
    """
    datos = obtener_datos(
        ciudad,
        tratamiento
    ).copy()

    datos = crear_capacidad_maxima(datos)

    datos = datos[
        (datos["Valoracion"] >= umbral)
        & (datos["Capacidad4Semanas"] > 0)
    ].copy()

    if datos.empty:
        return pd.DataFrame()

    # LEFT JOIN: un proveedor con CuotaObjetivo = 0 sigue visible.
    if cuotas_objetivo is not None and not cuotas_objetivo.empty:
        datos = datos.merge(
            cuotas_objetivo[
                ["CODPROVEEDOR", "CuotaObjetivo"]
            ],
            on="CODPROVEEDOR",
            how="left"
        )
    else:
        datos["CuotaObjetivo"] = 0.0

    datos["CuotaObjetivo"] = (
        datos["CuotaObjetivo"].fillna(0.0)
    )

    max_cuota = datos["CuotaObjetivo"].max()

    if max_cuota > 0:
        datos["ScoreCuotaMILP"] = (
            datos["CuotaObjetivo"] / max_cuota
        )
    else:
        datos["ScoreCuotaMILP"] = 0.0

    # Seguimiento real respecto al plan.
    total_real = sum(asignaciones_operativas.values())

    if incluir_desviacion and total_real > 0:
        datos["CuotaReal"] = datos["CODPROVEEDOR"].map(
            lambda x: asignaciones_operativas.get(x, 0)
            / total_real
        )

        datos["DesviacionPlan"] = (
            datos["CuotaObjetivo"]
            - datos["CuotaReal"]
        )

        minimo = datos["DesviacionPlan"].min()
        maximo = datos["DesviacionPlan"].max()

        if np.isclose(minimo, maximo):
            datos["ScoreDesviacion"] = 1.0
        else:
            datos["ScoreDesviacion"] = (
                datos["DesviacionPlan"] - minimo
            ) / (maximo - minimo)
    else:
        datos["CuotaReal"] = np.nan
        datos["DesviacionPlan"] = np.nan
        datos["ScoreDesviacion"] = np.nan

    datos["RankingWeb"] = calcular_indice(
        datos,
        peso_coste,
        peso_valoracion,
        peso_capacidad,
        peso_cuota
    )

    if incluir_desviacion and total_real > 0:
        # La desviación se incorpora como un ajuste pequeño del ranking,
        # manteniendo los cuatro pesos principales comparables.
        datos["RankingWeb"] = (
            0.95 * datos["RankingWeb"]
            + 0.05 * datos["ScoreDesviacion"].fillna(0.0)
        )

    datos = datos.sort_values(
        ["RankingWeb", "CosteMedio"],
        ascending=[False, True]
    )

    return datos[
        [
            "CODPROVEEDOR",
            "NOMBREPROVEEDOR",
            "CosteMedio",
            "Valoracion",
            "Capacidad4Semanas",
            "CapacidadRelativa",
            "CuotaObjetivo",
            "ScoreCoste",
            "ScoreCapacidad",
            "ScoreCuotaMILP",
            "CuotaReal",
            "DesviacionPlan",
            "RankingWeb"
        ]
    ]



Combinaciones ciudad-tratamiento-proveedor: 400
Combinaciones con capacidad anual conocida: 400
Combinaciones con capacidad anual ausente: 0


## Función web definitiva

In [18]:
def recomendar(
    ciudad,
    tratamiento,
    umbral=3.7
):

    datos = obtener_datos(
        ciudad,
        tratamiento
    )

    demanda = demanda_forecast(
        ciudad,
        tratamiento
    )

    cuotas = obtener_cuotas_objetivo(
        datos,
        demanda,
        umbral
    )

    ranking = obtener_ranking_web(
        ciudad,
        tratamiento,
        umbral,
        cuotas_objetivo=cuotas,
        peso_coste=0.25,
        peso_valoracion=0.45,
        peso_capacidad=0.20,
        peso_cuota=0.10
    )

    return ranking

## Prueba

In [21]:
def probar_ejemplo():

    ciudad = selector_ciudad.value
    tratamiento = selector_tratamiento.value

    ranking = recomendar(
        ciudad,
        tratamiento,
        3.7
    )

    return ranking



display(probar_ejemplo())


,CODPROVEEDOR,NOMBREPROVEEDOR,CosteMedio,Valoracion,Capacidad4Semanas,CapacidadRelativa,CuotaObjetivo,ScoreCoste,ScoreCapacidad,ScoreCuotaMILP,CuotaReal,DesviacionPlan,RankingWeb
1,3009,CLINICA LOS SAUCES C.A.,40.000,3.900,1.538,0.385,0.250,0.778,1.000,1.000,NaN,NaN,0.899
0,1134,CENTRO CLINICO FENIX SALUD C.A,49.000,4.000,1.538,0.385,0.250,0.578,1.000,1.000,NaN,NaN,0.894
